# Prompt Engineering pada Mistral 7B Instruct

Notebook ini digunakan untuk melakukan eksperimen prompt engineering pada model Mistral 7B Instruct tanpa fine-tuning. Prompt yang digunakan mengacu pada pedoman anotasi yang sama dengan proses pseudo-labeling menggunakan Gemini API. Penyesuaian yang dilakukan hanya bersifat teknis agar sesuai dengan mekanisme chat model Mistral 7B Instruct tanpa mengubah substansi pedoman anotasi.

Eksperimen dilakukan menggunakan tiga strategi prompting, yaitu zero-shot, one-shot, dan few-shot. Evaluasi dilakukan pada test set untuk mengukur kinerja model Mistral 7B Instruct tanpa fine-tuning pada masing-masing strategi prompting.

In [ ]:
# Pemeriksaan Environment Eksekusi
!nvidia-smi

In [ ]:
!pip install -q -U unsloth

In [ ]:
from unsloth import FastLanguageModel

import warnings
import os
import re
import json
import torch
import pandas as pd

from tqdm import tqdm
from transformers.utils import logging

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss
)

warnings.filterwarnings("ignore")
logging.set_verbosity_error()

In [ ]:
# Configuration and Data Loading
MODEL_NAME = "unsloth/mistral-7b-instruct-v0.3"

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
MAX_NEW_TOKENS = 64

DATA_DIR = "/kaggle/input/datasets/cindyzakyaandini/usability-datasets-v2"
OUTPUT_DIR = "/kaggle/working"

TEST_PATH = os.path.join(DATA_DIR, "test_ground_truth.csv")
SELECTED_EXAMPLES_PATH = os.path.join(DATA_DIR, "selected_prompt_examples.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

LABELS = [
    "accuracy",
    "completeness",
    "satisfaction"
]

In [ ]:
test_df = pd.read_csv(TEST_PATH)
selected_examples_df = pd.read_csv(SELECTED_EXAMPLES_PATH)

required_columns = [
    "cleaned_content",
    "accuracy",
    "completeness",
    "satisfaction"
]

for col in required_columns:
    if col not in test_df.columns:
        raise ValueError(f"Kolom '{col}' tidak ditemukan pada test_df")

required_example_columns = [
    "target_label",
    "cleaned_content",
    "accuracy",
    "completeness",
    "satisfaction"
]

for col in required_example_columns:
    if col not in selected_examples_df.columns:
        raise ValueError(f"Kolom '{col}' tidak ditemukan pada selected_examples_df")

print(f"Test data       : {len(test_df)}")
print(f"Selected examples: {len(selected_examples_df)}")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

FastLanguageModel.for_inference(model)

print("Model berhasil dimuat.")

In [ ]:
# Prompt Construction
def annotation_guideline():
    return """
    Anda adalah anotator usability.
    Tugas:
    Klasifikasikan setiap ulasan pengguna menggunakan multi-label classification ke dalam tiga label berikut:
    
    1. accuracy
    Label = 1 jika terdapat masalah yang menunjukkan kegagalan sistem, proses, transaksi, atau fungsi aplikasi dalam membantu pengguna mencapai tujuan penggunaan.
    Contoh:
    - error
    - bug
    - crash
    - force close
    - login gagal
    - transaksi gagal
    - OTP gagal
    - QRIS gagal
    - saldo atau dana hilang
    - pending
    - loading berkepanjangan
    - delay
    - verifikasi gagal
    - fitur tersedia tetapi tidak dapat digunakan
    - sistem menghasilkan hasil yang tidak sesuai harapan pengguna
    Fokus accuracy adalah fungsi aplikasi yang tidak berjalan sebagaimana mestinya.
    
    2. completeness
    Label = 1 jika pengguna mengeluhkan bahwa kebutuhan penggunaan belum didukung secara memadai oleh aplikasi.
    Contoh:
    - fitur tidak tersedia
    - fitur belum muncul
    - fitur kurang lengkap
    - membutuhkan fitur tambahan
    - layanan bantuan atau CS tidak membantu
    - proses upgrade akun dipersulit
    - opsi login atau akses dianggap tidak memadai
    - layanan atau fungsi yang dibutuhkan pengguna belum tersedia
    - proses penggunaan dianggap tidak lengkap atau tidak mendukung kebutuhan pengguna
    Fokus completeness adalah kelengkapan dukungan aplikasi terhadap kebutuhan pengguna.
    
    3. satisfaction
    Label = 1 jika terdapat ketidakpuasan, frustrasi, kekecewaan, kemarahan, keluhan bernada negatif, atau evaluasi negatif terhadap pengalaman penggunaan aplikasi.
    Contoh:
    - kecewa
    - kesal
    - marah
    - ribet
    - menyulitkan
    - membingungkan
    - buruk
    - parah
    - sangat mengecewakan
    Label satisfaction dapat diberikan meskipun tidak terdapat kata emosi secara eksplisit apabila isi ulasan secara keseluruhan menunjukkan pengalaman penggunaan yang negatif.
    
    Aturan penting:
    - Satu ulasan dapat memiliki lebih dari satu label.
    - Ulasan yang tidak berkaitan dengan usability dapat diberi semua label 0.
    - Label diberikan berdasarkan isi utama ulasan.
    - Fokus pada makna keseluruhan ulasan, bukan hanya keyword tertentu.
    - Keluhan teknis dapat menghasilkan accuracy = 1.
    - Keluhan teknis yang disertai nada negatif dapat menghasilkan satisfaction = 1.
    - Dana atau saldo hilang menghasilkan accuracy = 1.
    - Dana cicilan atau fitur yang belum muncul menghasilkan completeness = 1.
    - Fitur tersedia tetapi tidak dapat digunakan menghasilkan accuracy = 1.
    - CS atau layanan bantuan yang tidak membantu dapat dianggap completeness = 1 dan satisfaction = 1.
    - Jika suatu aspek tidak relevan, beri 0.
    
    Format output:
    {"accuracy":0,"completeness":0,"satisfaction":0}
    
    Output HARUS berupa JSON valid tanpa teks tambahan.
    """

In [ ]:
def load_prompt_examples(selected_examples_df):
    def select_examples(n_per_label):
        examples = []

        for label in LABELS:
            selected = (
                selected_examples_df.loc[
                    selected_examples_df["target_label"] == label
                ]
                .head(n_per_label)
            )

            if len(selected) != n_per_label:
                raise ValueError(
                    f"Jumlah contoh untuk label '{label}' harus {n_per_label}."
                )

            for _, row in selected.iterrows():
                examples.append({
                    "target_label": label,
                    "review": row["cleaned_content"],
                    "accuracy": int(row["accuracy"]),
                    "completeness": int(row["completeness"]),
                    "satisfaction": int(row["satisfaction"])
                })

        return examples

    one_shot_examples = select_examples(n_per_label=1)
    few_shot_examples = select_examples(n_per_label=2)

    return one_shot_examples, few_shot_examples


ONE_SHOT_EXAMPLES, FEW_SHOT_EXAMPLES = load_prompt_examples(selected_examples_df)

In [ ]:
def example_to_assistant_json(example):
    return (
        f'{{"accuracy":{int(example["accuracy"])},'
        f'"completeness":{int(example["completeness"])},'
        f'"satisfaction":{int(example["satisfaction"])}}}'
    )

def build_zero_shot_messages(review, guideline_func):
    return [
        {
            "role": "system",
            "content": guideline_func()
        },
        {
            "role": "user",
            "content": f"Ulasan:\n{review}"
        }
    ]


def build_one_shot_messages(review, guideline_func):
    messages = [
        {
            "role": "system",
            "content": guideline_func()
        }
    ]

    for example in ONE_SHOT_EXAMPLES:
        messages.append({
            "role": "user",
            "content": f'Ulasan:\n{example["review"]}'
        })
        messages.append({
            "role": "assistant",
            "content": example_to_assistant_json(example)
        })

    messages.append({
        "role": "user",
        "content": f"Ulasan:\n{review}"
    })

    return messages


def build_few_shot_messages(review, guideline_func):
    messages = [
        {
            "role": "system",
            "content": guideline_func()
        }
    ]

    for example in FEW_SHOT_EXAMPLES:
        messages.append({
            "role": "user",
            "content": f'Ulasan:\n{example["review"]}'
        })
        messages.append({
            "role": "assistant",
            "content": example_to_assistant_json(example)
        })

    messages.append({
        "role": "user",
        "content": f"Ulasan:\n{review}"
    })

    return messages

In [ ]:
# Inference and Output Parsing
def generate_response(review, guideline_func, prompt_builder):
    messages = prompt_builder(
        review=review,
        guideline_func=guideline_func
    )

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt"
    ).to(model.device)

    attention_mask = torch.ones_like(inputs)

    inputs = inputs.to(model.device)
    attention_mask = attention_mask.to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs.shape[-1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
def run_inference(
    df,
    guideline_func,
    prompt_builder,
    output_file,
    save_every=50
):
    result_df = df.copy()

    # Resume inference from the latest saved output
    if os.path.exists(output_file):
        saved_df = pd.read_csv(output_file)

        if len(saved_df) > len(result_df):
            raise ValueError(
                "Jumlah data pada file output lebih banyak daripada dataset input."
            )

        result_df.loc[:len(saved_df)-1, LABELS] = saved_df[LABELS].values
        result_df.loc[:len(saved_df)-1, "raw_output"] = saved_df["raw_output"].values

        start_idx = len(saved_df)

        print(f"Melanjutkan dari index {start_idx}")
    else:
        result_df["raw_output"] = ""
        for label in LABELS:
            result_df[label] = None

        start_idx = 0

    for idx in tqdm(
        range(start_idx, len(result_df)),
        total=len(result_df) - start_idx
    ):
        review = result_df.loc[idx, "cleaned_content"]

        raw_output = generate_response(
            review=review,
            guideline_func=guideline_func,
            prompt_builder=prompt_builder
        )

        parsed_output = parse_json_output(raw_output)

        for label in LABELS:
            result_df.loc[idx, label] = parsed_output[label]

        result_df.loc[idx, "raw_output"] = raw_output

        # Save intermediate results periodically
        if (idx + 1) % save_every == 0:
            result_df.iloc[:idx + 1].to_csv(
                output_file,
                index=False,
                encoding="utf-8"
            )
            print(f"Autosaved sampai index {idx}")

    # Save the complete prediction results
    result_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8"
    )

    print(f"Hasil prediksi disimpan ke: {output_file}")

    return result_df

In [ ]:
def parse_json_output(text):
    default_output = {
        "accuracy": 0,
        "completeness": 0,
        "satisfaction": 0
    }

    try:
        text = str(text).strip()

        text = text.replace("```json", "")
        text = text.replace("```", "")

        match = re.search(
            r"\{[^{}]*\}",
            text,
            re.DOTALL
        )

        if not match:
            return default_output

        parsed = json.loads(
            match.group(0)
        )

        result = {}

        for label in LABELS:
            value = parsed.get(label, 0)

            if value in [1, "1", True, "true", "True"]:
                result[label] = 1
            else:
                result[label] = 0

        return result

    except Exception:
        return default_output

In [ ]:
def evaluate_predictions(ground_truth_df, prediction_df):
    y_true = (
        ground_truth_df[LABELS]
        .fillna(0)
        .astype(int)
    )

    y_pred = (
        prediction_df[LABELS]
        .fillna(0)
        .astype(int)
    )

    results = {
        "macro_precision": round(precision_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "macro_recall": round(recall_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "macro_f1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "micro_precision": round(precision_score(y_true, y_pred, average="micro", zero_division=0), 4),
        "micro_recall": round(recall_score(y_true, y_pred, average="micro", zero_division=0), 4),
        "micro_f1": round(f1_score(y_true, y_pred,average="micro", zero_division=0), 4),
        "subset_accuracy": round(accuracy_score(y_true, y_pred), 4),
        "hamming_loss": round(hamming_loss(y_true, y_pred), 4)
    }

    for label in LABELS:
        results[f"{label}_precision"] = round(
            precision_score(
                y_true[label],
                y_pred[label],
                zero_division=0
            ),
            4
        )

        results[f"{label}_recall"] = round(
            recall_score(
                y_true[label],
                y_pred[label],
                zero_division=0
            ),
            4
        )

        results[f"{label}_f1"] = round(
            f1_score(
                y_true[label],
                y_pred[label],
                zero_division=0
            ),
            4
        )

    return results

In [ ]:
TEST_EXPERIMENTS = [
    {
        "prompting_method": "zero_shot",
        "guideline_func": annotation_guideline,
        "prompt_builder": build_zero_shot_messages,
        "output_file": "test_zero_shot.csv"
    },
    {
        "prompting_method": "one_shot",
        "guideline_func": annotation_guideline,
        "prompt_builder": build_one_shot_messages,
        "output_file": "test_one_shot.csv"
    },
    {
        "prompting_method": "few_shot",
        "guideline_func": annotation_guideline,
        "prompt_builder": build_few_shot_messages,
        "output_file": "test_few_shot.csv"
    },
]

test_results = []

for exp in TEST_EXPERIMENTS:
    print("=" * 80)
    print(f"Test | {exp['prompting_method']}")
    print("=" * 80)

    output_path = os.path.join(
        OUTPUT_DIR,
        exp["output_file"]
    )

    pred_df = run_inference(
        df=test_df,
        guideline_func=exp["guideline_func"],
        prompt_builder=exp["prompt_builder"],
        output_file=output_path,
        save_every=50
    )

    metrics = evaluate_predictions(
        ground_truth_df=test_df,
        prediction_df=pred_df
    )

    test_results.append({
        "model": "mistral_base",
        "dataset": "test",
        "prompting_method": exp["prompting_method"],
        **metrics
    })

test_comparison_df = pd.DataFrame(test_results)

test_comparison_output = os.path.join(
    OUTPUT_DIR,
    "test_base_prompting_comparison.csv"
)

test_comparison_df.to_csv(
    test_comparison_output,
    index=False,
    encoding="utf-8"
)

test_comparison_df